# Task 1: Classifying Parkinson's vs control
Run this notebook to predict whether a person has Parkinson's disease (PD) from their gait features.

The structure of this notebook is as follows:
1. Data setup: loading features, labels, splits
2. Training helper: function to train models and save outputs
3. Majority baseline
4. Logistic regression
5. Random forest
6. Results: comparing the models
7. 1D-CNN (to be added)

## 1. Data setup

In [1]:
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))

# build data/processed/ if it is missing (features.py needs it)
from data import txt_to_csv, raw_dir, out_dir
if not out_dir.exists():
    txt_to_csv(raw_dir, out_dir)

SPLITS_PATH = ROOT / "data" / "splits.json"

# output directory
OUT = ROOT / "presentation_material" / "task1"
(OUT / "predictions").mkdir(parents=True, exist_ok=True)
(OUT / "weights").mkdir(parents=True, exist_ok=True)

# import functions from models
from features import X, y
from src.models.logreg import make_logistic_regression
from src.models.rf import make_random_forest

        stride_time  stride_cv  swing_cv  swing_asymmetry  peak_force
GaCo01     1.230322   0.042125  0.054223         0.013902   14.561747
GaCo02     1.103386   0.035667  0.037527         0.001486   15.233429
GaCo03     1.241191   0.049523  0.078598         0.103336   12.754554
GaCo04     1.249965   0.058603  0.064637         0.022956   15.401692
GaCo05     1.051882   0.039561  0.055866         0.013298   12.820625
...             ...        ...       ...              ...         ...
SiPt36     1.065997   0.041058  0.077181         0.066450   14.236129
SiPt37     1.210989   0.056895  0.072182         0.011100   12.766210
SiPt38     1.163753   0.039291  0.049182         0.000858   12.365833
SiPt39     1.067753   0.048137  0.056888         0.065241   21.129340
SiPt40     1.005855   0.045557  0.056307         0.018599   13.568025

[165 rows x 5 columns]
GaCo01    0
GaCo02    0
GaCo03    0
GaCo04    0
GaCo05    0
         ..
SiPt36    1
SiPt37    1
SiPt38    1
SiPt39    1
SiPt40    1
Name

In [2]:
# X = one row per subject with 5 gait features, y = 1 if Parkinson's, 0 if healthy control
# (both come from features.py, so no need to merge metadata here)
print("Subjects:", X.shape[0], "| Feature matrix:", X.shape, "| PD share:", round(y.mean(), 3))
display(X.head())

Subjects: 165 | Feature matrix: (165, 5) | PD share: 0.564


,stride_time,stride_cv,swing_cv,swing_asymmetry,peak_force
GaCo01,1.230322,0.042125,0.054223,0.013902,14.561747
GaCo02,1.103386,0.035667,0.037527,0.001486,15.233429
GaCo03,1.241191,0.049523,0.078598,0.103336,12.754554
GaCo04,1.249965,0.058603,0.064637,0.022956,15.401692
GaCo05,1.051882,0.039561,0.055866,0.013298,12.820625


In [3]:
with open(SPLITS_PATH) as f:
    assignment = json.load(f)["task1"]

test_subjects = set(assignment["test"])
folds = []
for fold in assignment["folds"]:
    train_ids, val_ids = set(fold["train"]), set(fold["val"])
    # Essential safety check: keep the held-out test subjects separate.
    assert train_ids.isdisjoint(val_ids | test_subjects)
    assert val_ids.isdisjoint(test_subjects)
    train_idx = np.flatnonzero(X.index.isin(train_ids))
    val_idx = np.flatnonzero(X.index.isin(val_ids))
    folds.append((train_idx, val_idx))

print("Using", len(folds), "saved validation folds;", len(test_subjects), "held-out test subjects untouched.")

Using 5 saved validation folds; 33 held-out test subjects untouched.


## 2. Training helper

Function to implement at every model to train the model and save the output.
For each fold it trains on the training subjects only, predicts the probability of PD for the validation subjects, and saves the predictions and the fitted model.

In [4]:
def train_and_save(model_name, make_model):
    for k, (tr, va) in enumerate(folds, start = 1):
        model = make_model() # new, unfitted model per fold
        model.fit(X.iloc[tr], y.iloc[tr])
        prob = model.predict_proba(X.iloc[va])[:, 1]   # probability of PD
        pd.DataFrame({
            "subject": X.index[va],
            "fold": k,
            "y_true": y.iloc[va].to_numpy(),
            "y_pred": (prob >= 0.5).astype(int),
            "y_prob": prob,
        }).to_csv(
            OUT / "predictions" / f"{model_name}_val_fold{k}.csv", index = False
        )
        # save fitted preprocessing steps and model together.
        joblib.dump(
            model, OUT / "weights" / f"{model_name}_val_fold{k}.joblib"
        )

    print(f"{model_name}: model train done")

## 3. Majority baseline
This is used as a reference point. About 56% of subjects are classified correctly and the ROC-AUC is 0.5.

In [5]:
# "prior" always predicts the training-fold majority class (PD) and gives a constant probability
train_and_save("majority_baseline", lambda: DummyClassifier(strategy="prior"))

majority_baseline: model train done


## 4. Logistic Regression
Weighs each gait feature and turns the total into a probability of PD. Features are standardised first (see `src/models/logreg.py`).

In [6]:
train_and_save("logistic_regression", make_logistic_regression)

logistic_regression: model train done


## 5. Random Forest

In [7]:
train_and_save("random_forest", lambda: make_random_forest(task="classification"))

random_forest: model train done


## 7. 1D-CNN Classification

## 6. Results
Scores on the validation folds, shown as mean ± standard deviation over the 5 folds.

In [8]:
def fold_metrics(name):
    rows = []
    for k in range(1, len(folds) + 1):
        d = pd.read_csv(OUT / "predictions" / f"{name}_val_fold{k}.csv")
        rows.append({"accuracy": accuracy_score(d.y_true, d.y_pred),
                     "f1": f1_score(d.y_true, d.y_pred),
                     "roc_auc": roc_auc_score(d.y_true, d.y_prob)})
    return pd.DataFrame(rows)

summary = {}
for name in ["majority_baseline", "logistic_regression", "random_forest"]:
    m = fold_metrics(name)
    summary[name] = {f"{c}": f"{m[c].mean():.3f} ± {m[c].std():.3f}" for c in m}
pd.DataFrame(summary).T

,accuracy,f1,roc_auc
majority_baseline,0.561 ± 0.016,0.718 ± 0.013,0.500 ± 0.000
logistic_regression,0.750 ± 0.066,0.773 ± 0.046,0.769 ± 0.056
random_forest,0.727 ± 0.051,0.753 ± 0.035,0.744 ± 0.075
